# Mapeamento: df_final.csv ⇄ db.json (por título)

Este notebook tenta mapear os hinos do `hinos-musicas/df_final.csv` com os itens do `db.json` (raiz) **sem um ID comum**, usando normalização de texto + matching exato + matching fuzzy (RapidFuzz) e, quando necessário, um fallback com LLM.

## Estratégia
- 1) `exact`: tenta match exato pelo nome do hino e também pela combinação `album_name | title`.
- 2) `fuzzy`: tenta match por similaridade (RapidFuzz). Só **aceita automaticamente** quando `score >= 95`.
- 3) `llm`: para o restante (score < 95 ou sem match), consulta um modelo barato (ex.: `gpt-4o-mini`) com uma lista curta de candidatos, e o modelo pode responder **"sem match"** quando não estiver seguro.

## Saídas
- `songs_enriched_from_df_final.csv`: base no `df_final.csv` + campos enriquecidos vindos do `db.json` (`id_igcg`, `title_igcg`, `thumbnail`, `file_url`, `file_type`, `file_duration`, `members`, `description`).
- `title_mapping_report.csv`: relatório de mapeamento (título df_final, título igcg, score, método).

## API Key (OpenAI)
- A chave deve estar no `.env` como `API_KEY_OPENAI` (o notebook tenta carregar `.env`).
- Não coloque chaves diretamente no notebook/repositório. Se uma chave vazou, revogue e gere outra.

In [19]:
# Se precisar instalar dependências no kernel do notebook, rode (uma vez):
# %pip install -U pandas rapidfuzz

import json
import os
import re
import time
import unicodedata
from pathlib import Path
from typing import Any
import pandas as pd

try:
    from rapidfuzz import process, fuzz
except Exception as e:
    raise RuntimeError("Faltou rapidfuzz. Rode: %pip install rapidfuzz") from e

ROOT = Path('.').resolve()
DF_FINAL_PATH = ROOT / 'hinos-musicas' / 'df_final.csv'
DB_JSON_PATH = ROOT / 'db.json'
ENV_PATH = ROOT / '.env'

assert DF_FINAL_PATH.exists(), f'Não achei: {DF_FINAL_PATH}'
assert DB_JSON_PATH.exists(), f'Não achei: {DB_JSON_PATH}'

def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        # Não sobrescreve se já existir no ambiente
        os.environ.setdefault(k, v)

load_env_file(ENV_PATH)

AUTO_ACCEPT_SCORE = 95
LLM_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
LLM_MIN_CONFIDENCE = float(os.getenv('LLM_MIN_CONFIDENCE', '0.75'))
MAX_LLM_CALLS = int(os.getenv('MAX_LLM_CALLS', '300'))
OPENAI_API_KEY = os.getenv('API_KEY_OPENAI')

print('DF_FINAL_PATH:', DF_FINAL_PATH)
print('DB_JSON_PATH:', DB_JSON_PATH)
print('AUTO_ACCEPT_SCORE:', AUTO_ACCEPT_SCORE)
print('LLM_MODEL:', LLM_MODEL)
print('MAX_LLM_CALLS:', MAX_LLM_CALLS)
print('Has API_KEY_OPENAI:', bool(OPENAI_API_KEY))

DF_FINAL_PATH: C:\Users\sword\projetos\backend_igcg\hinos-musicas\df_final.csv
DB_JSON_PATH: C:\Users\sword\projetos\backend_igcg\db.json
AUTO_ACCEPT_SCORE: 95
LLM_MODEL: gpt-4o-mini
MAX_LLM_CALLS: 300
Has API_KEY_OPENAI: True


In [20]:
df = pd.read_csv(DF_FINAL_PATH)
print('df_final shape:', df.shape)
print('df_final columns:', list(df.columns))
df.head(3)

df_final shape: (1307, 14)
df_final columns: ['id', 'slug', 'link', 'title', 'album_name', 'date', 'modified', 'categories', 'tags', 'sheet_links', 'ui_fields', 'lyrics_chords_text', 'lyrics', 'language']


,id,slug,link,title,album_name,date,modified,categories,tags,sheet_links,ui_fields,lyrics_chords_text,lyrics,language
0,14166,10-tu-eres-mas-288,https://timcifras.com.br/musicas/10-tu-eres-ma...,Tú Eres Más,288,2023-04-11 20:12:41,2023-05-01 18:22:47,"[{'id': 82, 'name': 'Músicas 288', 'slug': 'mu...",[],{'pdf_view_url': 'https://timcifras.com.br/?je...,"{'rhythm': 'Canção', 'original_key_ui': 'G', '...",[G]Tú eres [B7]más de lo [C]que pensé De lo q...,NaN,espanhol
1,14429,whenever-im-low,https://timcifras.com.br/musicas/whenever-im-low/,Whenever I’m low,288,2023-04-11 20:14:07,2023-04-30 20:26:48,"[{'id': 82, 'name': 'Músicas 288', 'slug': 'mu...",[],{'pdf_view_url': 'https://timcifras.com.br/?je...,"{'rhythm': 'Folk', 'original_key_ui': 'C', 'in...",1.[C]You can be the love in my heart; [F]You c...,NaN,inglês
2,17173,nao-ha-ninguem-igual-pra-mim-cd-oh-deus-abenco...,https://timcifras.com.br/musicas/nao-ha-ningue...,Não há ninguém igual pra mim,Oh Deus abençoe a Àfrica,2025-09-23 20:57:54,2025-09-23 20:57:54,"[{'id': 83, 'name': 'Hinos dos CDs', 'slug': '...",[],{'pdf_view_url': 'https://timcifras.com.br/?je...,"{'rhythm': 'Reggae', 'original_key_ui': 'G', '...","[G]Do Senhor o que direi, não há, ninguém igua...",NaN,português


In [21]:
with DB_JSON_PATH.open('r', encoding='utf-8') as f:
    db_data = json.load(f)

# db.json parece ter a estrutura: { "slug": [ {id,title,members,thumbnail,description,file:{url,type,duration},...}, ... ] }
if isinstance(db_data, dict) and len(db_data) == 1:
    only_key = next(iter(db_data.keys()))
    records = db_data[only_key]
elif isinstance(db_data, dict) and 'slug' in db_data:
    records = db_data['slug']
elif isinstance(db_data, list):
    records = db_data
else:
    raise ValueError('Estrutura inesperada de db.json')

igcg = pd.json_normalize(records)
print('db.json records:', len(records))
print('igcg columns:', list(igcg.columns))
igcg.head(3)

db.json records: 1352
igcg columns: ['id', 'title', 'members', 'published_at', 'thumbnail', 'description', 'file.url', 'file.type', 'file.duration']


,id,title,members,published_at,thumbnail,description,file.url,file.type,file.duration
0,cdvlrdgrdamor-a-bencaodeabraao,CD O Valor de Um Grande Amor | A Benção de Abraão,Tim Herbert,2021-09-09 13:00:00,https://igcgcloud.netlify.app/images/capaovalo...,,https://igcgcloud.netlify.app/cd-ovalordeumgra...,audio/mp3,178
1,cdvlrdgrdamor-de-barrabasafilho,CD O Valor de Um Grande Amor | De Barrabás a F...,Tim Herbert,2021-09-09 13:00:00,https://igcgcloud.netlify.app/images/capaovalo...,,https://igcgcloud.netlify.app/cd-ovalordeumgra...,audio/mp3,239
2,cdvlrdgrdamor-de-voltaaoinicio-tim,CD O Valor de Um Grande Amor | De Volta ao Inicio,Tim Herbert,2021-09-09 13:00:00,https://igcgcloud.netlify.app/images/capaovalo...,,https://igcgcloud.netlify.app/cd-ovalordeumgra...,audio/mp3,189


In [28]:
def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFKD', text)
    return ''.join(ch for ch in text if not unicodedata.combining(ch))

def normalize_title(text: str) -> str:
    if text is None:
        return ''
    text = str(text).strip()
    if not text:
        return ''
    text = strip_accents(text)
    text = text.lower()
    text = text.replace('\u00a0', ' ')
    # preserva '|' como separador útil
    text = re.sub(r"[^a-z0-9\s|]+", ' ', text)
    text = re.sub(r"\s+", ' ', text).strip()
    return text

def split_igcg_title(full_title: str) -> tuple[str, str]:
    """Retorna (album_part, hymn_part) a partir de 'Album | Hino'."""
    if full_title is None:
        return ('', '')
    full_title = str(full_title).strip()
    if '|' in full_title:
        left, right = full_title.split('|', 1)
        return (left.strip(), right.strip())
    return ('', full_title)

def make_combo_norm(album: str, title: str) -> str:
    album_n = normalize_title(album)
    title_n = normalize_title(title)
    if album_n and title_n:
        return f"{album_n} | {title_n}"
    return title_n or album_n

df['title_norm'] = df['title'].map(normalize_title)
df['album_norm'] = df.get('album_name', '').map(normalize_title)
df['combo_norm'] = df.apply(lambda r: make_combo_norm(r.get('album_name',''), r.get('title','')), axis=1)

igcg['title_igcg'] = igcg.get('title', '').astype(str)
igcg[['album_igcg', 'hymn_name']] = igcg['title_igcg'].apply(lambda t: pd.Series(split_igcg_title(t)))
igcg['album_igcg_norm'] = igcg['album_igcg'].map(normalize_title)
igcg['hymn_name_norm'] = igcg['hymn_name'].map(normalize_title)
igcg['combo_norm'] = igcg.apply(lambda r: make_combo_norm(r.get('album_igcg',''), r.get('hymn_name','')), axis=1)

print('df_final normalized title empty:', (df['title_norm'] == '').sum())
print('df_final normalized combo empty:', (df['combo_norm'] == '').sum())
print('igcg normalized hymn_name empty:', (igcg['hymn_name_norm'] == '').sum())
print('igcg normalized combo empty:', (igcg['combo_norm'] == '').sum())
df[['title','album_name','title_norm','combo_norm']].head(5)

df_final normalized title empty: 0
df_final normalized combo empty: 0
igcg normalized hymn_name empty: 0
igcg normalized combo empty: 0


,title,album_name,title_norm,combo_norm
0,Tú Eres Más,288,tu eres mas,288 | tu eres mas
1,Whenever I’m low,288,whenever i m low,288 | whenever i m low
2,Não há ninguém igual pra mim,Oh Deus abençoe a Àfrica,nao ha ninguem igual pra mim,oh deus abencoe a africa | nao ha ninguem igua...
3,Antes de Haver o Tempo,NaN,antes de haver o tempo,nan | antes de haver o tempo
4,Deus é fiel,Nos Braços do Pai,deus e fiel,nos bracos do pai | deus e fiel


In [29]:
# 1) Matching exato (prioriza combo album|title; senão, só title)
#
# Importante: db.json pode ter títulos repetidos (mesmo hino em mais de um lugar).
# Para não duplicar linhas no merge, deduplicamos as chaves no lado do db.json de forma determinística.
igcg_keyed_cols = [
    'id',
    'title_igcg',
    'album_igcg',
    'hymn_name',
    'album_igcg_norm',
    'hymn_name_norm',
    'combo_norm',
    'members',
    'description',
    'thumbnail',
    'file.url',
    'file.type',
    'file.duration',
]
if 'published_at' in igcg.columns:
    igcg_keyed_cols.append('published_at')

igcg_keyed = igcg[igcg_keyed_cols].copy()
igcg_keyed = igcg_keyed.rename(columns={
    'id': 'id_igcg',
    'file.url': 'file_url',
    'file.type': 'file_type',
    'file.duration': 'file_duration',
})

# Ordenação para escolher "o melhor" quando a chave repete
sort_cols = []
sort_asc = []
if 'published_at' in igcg_keyed.columns:
    # publicado mais recente primeiro
    sort_cols.append('published_at')
    sort_asc.append(False)
sort_cols.append('id_igcg')
sort_asc.append(False)

igcg_sorted = igcg_keyed.sort_values(sort_cols, ascending=sort_asc, na_position='last')

# Dedup por combo e por title (hymn_name_norm)
igcg_by_combo = igcg_sorted.dropna(subset=['combo_norm']).drop_duplicates(subset=['combo_norm'], keep='first').copy()
igcg_by_title = igcg_sorted.dropna(subset=['hymn_name_norm']).drop_duplicates(subset=['hymn_name_norm'], keep='first').copy()

# Passo A: match exato por combo_norm (album|title)
exact = df.join(
    igcg_by_combo.set_index('combo_norm'),
    on='combo_norm',
    rsuffix='_igcg',
)
exact['match_method'] = exact['id_igcg'].apply(lambda x: 'exact_combo' if pd.notna(x) else None)
exact['match_score'] = exact['id_igcg'].apply(lambda x: 100 if pd.notna(x) else None)

# Passo B: onde não achou, match exato por título do hino (sem album)
fallback = df.join(
    igcg_by_title.set_index('hymn_name_norm'),
    on='title_norm',
    rsuffix='_igcg_t',
)

mask = exact['id_igcg'].isna()
cols_to_fill = [c for c in igcg_by_title.columns if c in fallback.columns]
for c in cols_to_fill:
    exact.loc[mask, c] = exact.loc[mask, c].combine_first(fallback.loc[mask, c])

# define método/score para os que preencheram via exact_title
filled_title = mask & exact['id_igcg'].notna()
exact.loc[filled_title, 'match_method'] = 'exact_title'
exact.loc[filled_title, 'match_score'] = 100

matched_exact = exact['id_igcg'].notna().sum()
print('Matched exact (combo+title):', matched_exact, 'of', len(exact))
exact[['title','album_name','id_igcg','title_igcg','hymn_name','match_method']].head(10)

Matched exact (combo+title): 338 of 1307


,title,album_name,id_igcg,title_igcg,hymn_name,match_method
0,Tú Eres Más,288,288-tu-eres-mas,CD 288 | Tú Eres Más,Tú Eres Más,exact_title
1,Whenever I’m low,288,288-whenever-im-low,CD 288 | Whenever I'm Low,Whenever I'm Low,exact_title
2,Não há ninguém igual pra mim,Oh Deus abençoe a Àfrica,daaa-nao-ha-ninguem-igual-pra-mim,CD Deus Abençoe a África | Não Há Nínguem Igua...,Não Há Nínguem Igual Pra Mim,exact_title
3,Antes de Haver o Tempo,NaN,288-antes-de-haver-o-tempo,CD 288 | Antes de Haver o Tempo,Antes de Haver o Tempo,exact_title
4,Deus é fiel,Nos Braços do Pai,NaN,NaN,NaN,None
5,Jeremias 29-13,FVK,NaN,NaN,NaN,None
6,Jovem onde estás?,Carta Viva,cv-jovem-onde-estas,CD Carta Viva | Jovem Onde Estás,Jovem Onde Estás,exact_title
7,Nossa Missão,NaN,NaN,NaN,NaN,None
8,Encontro de Glorias,NaN,NaN,NaN,NaN,None
9,Tomando a Palavra,O Fluir do Espírito,cdofde-tomando-a-palavra,CD O Fluir do Espírito | Tomando a Palavra,Tomando a Palavra,exact_title


In [34]:
# 2) Fuzzy matching (somente para não casados no passo exato)
# Só aceita automaticamente quando score >= AUTO_ACCEPT_SCORE (95)
igcg_candidates = igcg_keyed.dropna(subset=['hymn_name_norm']).copy()

# OBS: id_igcg no db.json é string (slug), não int.
igcg_candidates['id_igcg_str'] = igcg_candidates['id_igcg'].astype(str)

# Para evitar colisões por strings duplicadas, cria uma choice string única por id
igcg_candidates['choice_combo'] = igcg_candidates.apply(lambda r: f"{r['combo_norm']}:::{r['id_igcg_str']}", axis=1)
igcg_candidates['choice_title'] = igcg_candidates.apply(lambda r: f"{r['hymn_name_norm']}:::{r['id_igcg_str']}", axis=1)

choice_to_row = {
    str(r['id_igcg_str']): r
    for _, r in igcg_candidates.iterrows()
}

def extract_best_choice(query: str, use_combo: bool) -> tuple[str | None, float | None]:
    if not query:
        return (None, None)
    choices = igcg_candidates['choice_combo'] if use_combo else igcg_candidates['choice_title']
    match = process.extractOne(query, choices, scorer=fuzz.WRatio)
    if not match:
        return (None, None)
    choice_str, score, _ = match
    try:
        id_part = str(choice_str.split(':::', 1)[1])
    except Exception:
        return (None, None)
    return (id_part, float(score))

need_fuzzy = exact[exact['id_igcg'].isna()].copy()
print('Need fuzzy:', len(need_fuzzy))

fuzzy_rows = []
for i, row in need_fuzzy.iterrows():
    query_combo = row.get('combo_norm', '')
    query_title = row.get('title_norm', '')
    # tenta combo primeiro (quando album existe), senão title
    use_combo = bool(row.get('album_norm', ''))
    best_id, best_score = extract_best_choice(query_combo if use_combo else query_title, use_combo=use_combo)
    if best_id is None:
        fuzzy_rows.append({'index': i, 'id_igcg_fuzzy': None, 'match_score_fuzzy': None, 'fuzzy_used': 'combo' if use_combo else 'title'})
        continue
    fuzzy_rows.append({'index': i, 'id_igcg_fuzzy': best_id, 'match_score_fuzzy': best_score, 'fuzzy_used': 'combo' if use_combo else 'title'})

fuzzy_df = pd.DataFrame(fuzzy_rows).set_index('index')
print('Fuzzy produced rows:', len(fuzzy_df))
fuzzy_df.head(5)

Need fuzzy: 969
Fuzzy produced rows: 969


,id_igcg_fuzzy,match_score_fuzzy,fuzzy_used
index,,,
4,cdvlrdgrdamor-a-bencaodeabraao,85.5,combo
5,cdvlrdgrdamor-a-bencaodeabraao,85.5,combo
7,cdvlrdgrdamor-a-bencaodeabraao,85.5,combo
8,cdvlrdgrdamor-a-bencaodeabraao,85.5,combo
10,cdvlrdgrdamor-a-bencaodeabraao,85.5,combo


In [35]:
# 3) Consolida: mantém exact; aplica fuzzy somente quando score >= AUTO_ACCEPT_SCORE; resto vai para LLM
import numpy as np

final_df = exact.copy()

# aplica fuzzy nas linhas sem match exato
for idx, frow in fuzzy_df.iterrows():
    fid = frow.get('id_igcg_fuzzy')
    fscore = frow.get('match_score_fuzzy')
    if fid is None or fscore is None:
        continue
    if fscore < AUTO_ACCEPT_SCORE:
        continue
    r = choice_to_row.get(str(fid))
    if r is None:
        continue
    final_df.loc[idx, 'id_igcg'] = r.get('id_igcg')
    final_df.loc[idx, 'title_igcg'] = r.get('title_igcg')
    final_df.loc[idx, 'album_igcg'] = r.get('album_igcg')
    final_df.loc[idx, 'hymn_name'] = r.get('hymn_name')
    final_df.loc[idx, 'members'] = r.get('members')
    final_df.loc[idx, 'description'] = r.get('description')
    final_df.loc[idx, 'thumbnail'] = r.get('thumbnail')
    final_df.loc[idx, 'file_url'] = r.get('file_url')
    final_df.loc[idx, 'file_type'] = r.get('file_type')
    final_df.loc[idx, 'file_duration'] = r.get('file_duration')
    final_df.loc[idx, 'match_method'] = 'fuzzy'
    final_df.loc[idx, 'match_score'] = fscore

# marca restantes como pendentes para LLM (não força)
pending_llm_mask = final_df['id_igcg'].isna()
final_df.loc[pending_llm_mask, 'match_method'] = final_df.loc[pending_llm_mask, 'match_method'].fillna('pending_llm')
final_df.loc[pending_llm_mask, 'match_score'] = np.nan

print('Matched after exact+fuzzy>=', AUTO_ACCEPT_SCORE, ':', final_df['id_igcg'].notna().sum(), 'of', len(final_df))
print('Pending LLM:', (final_df['match_method'] == 'pending_llm').sum())
final_df[['title','album_name','id_igcg','title_igcg','match_method','match_score']].head(10)

Matched after exact+fuzzy>= 95 : 341 of 1307
Pending LLM: 966


,title,album_name,id_igcg,title_igcg,match_method,match_score
0,Tú Eres Más,288,288-tu-eres-mas,CD 288 | Tú Eres Más,exact_title,100.0
1,Whenever I’m low,288,288-whenever-im-low,CD 288 | Whenever I'm Low,exact_title,100.0
2,Não há ninguém igual pra mim,Oh Deus abençoe a Àfrica,daaa-nao-ha-ninguem-igual-pra-mim,CD Deus Abençoe a África | Não Há Nínguem Igua...,exact_title,100.0
3,Antes de Haver o Tempo,NaN,288-antes-de-haver-o-tempo,CD 288 | Antes de Haver o Tempo,exact_title,100.0
4,Deus é fiel,Nos Braços do Pai,NaN,NaN,pending_llm,NaN
5,Jeremias 29-13,FVK,NaN,NaN,pending_llm,NaN
6,Jovem onde estás?,Carta Viva,cv-jovem-onde-estas,CD Carta Viva | Jovem Onde Estás,exact_title,100.0
7,Nossa Missão,NaN,NaN,NaN,pending_llm,NaN
8,Encontro de Glorias,NaN,NaN,NaN,pending_llm,NaN
9,Tomando a Palavra,O Fluir do Espírito,cdofde-tomando-a-palavra,CD O Fluir do Espírito | Tomando a Palavra,exact_title,100.0


In [36]:
# 4) LLM fallback: tenta resolver apenas o que ficou pendente (sem forçar match)
import urllib.request
import urllib.error

def openai_chat_json(model: str, api_key: str, system: str, user: str, timeout: int = 60) -> dict[str, Any]:
    url = 'https://api.openai.com/v1/chat/completions'
    payload = {
        'model': model,
        'temperature': 0.0,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        'response_format': {'type': 'json_object'},
    }
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        url,
        data=data,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {api_key}',
        },
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            body = resp.read().decode('utf-8')
            return json.loads(body)
    except urllib.error.HTTPError as e:
        detail = e.read().decode('utf-8', errors='ignore')
        raise RuntimeError(f'OpenAI HTTPError {e.code}: {detail[:500]}')
    except Exception as e:
        raise RuntimeError(f'OpenAI request failed: {e}')

def llm_pick_candidate(row: pd.Series, candidates: list[dict[str, Any]]) -> dict[str, Any]:
    """Retorna {match_id_igcg: str|None, confidence: float, reason: str}."""
    album = str(row.get('album_name', '') or '')
    title = str(row.get('title', '') or '')
    system = (
        'Você é um assistente de data-matching. Sua tarefa é decidir se um item A (df_final) '
        'é o mesmo hino que um item B (db.json) com base em título e album. '
        'Responda SOMENTE JSON com as chaves: match_id_igcg (string ou null), confidence (0 a 1), reason (string curta). '
        'Se não houver candidato claramente correspondente, retorne match_id_igcg=null com baixa confidence.'
    )
    cand_lines = []
    for c in candidates:
        cand_lines.append({
            'id_igcg': c.get('id_igcg'),
            'title_igcg': c.get('title_igcg'),
            'album_igcg': c.get('album_igcg'),
            'hymn_name': c.get('hymn_name'),
            'fuzzy_score': c.get('fuzzy_score'),
        })
    user = json.dumps({
        'df_final': {'album_name': album, 'title': title},
        'candidates': cand_lines,
        'notes': 'No db.json, title_igcg normalmente é "album | hino". Só escolha se estiver bem confiante.',
    }, ensure_ascii=False)
    raw = openai_chat_json(LLM_MODEL, OPENAI_API_KEY, system=system, user=user)
    content = raw.get('choices', [{}])[0].get('message', {}).get('content', '')
    try:
        parsed = json.loads(content)
    except Exception:
        parsed = {}
    return parsed

def top_candidates_for_row(row: pd.Series, limit: int = 8) -> list[dict[str, Any]]:
    """Pré-filtra candidatos via fuzzy (barato), para não mandar lista gigante ao LLM."""
    use_combo = bool(row.get('album_norm', ''))
    query = row.get('combo_norm', '') if use_combo else row.get('title_norm', '')
    if not query:
        return []
    choices = igcg_candidates['choice_combo'] if use_combo else igcg_candidates['choice_title']
    extracted = process.extract(query, choices, scorer=fuzz.WRatio, limit=limit)
    out: list[dict[str, Any]] = []
    for choice_str, score, _ in extracted:
        try:
            id_part = str(choice_str.split(':::', 1)[1])
        except Exception:
            continue
        r = choice_to_row.get(id_part)
        if r is None:
            continue
        out.append({
            'id_igcg': str(r.get('id_igcg')),
            'title_igcg': r.get('title_igcg'),
            'album_igcg': r.get('album_igcg'),
            'hymn_name': r.get('hymn_name'),
            'thumbnail': r.get('thumbnail'),
            'file_url': r.get('file_url'),
            'file_type': r.get('file_type'),
            'file_duration': r.get('file_duration'),
            'members': r.get('members'),
            'description': r.get('description'),
            'fuzzy_score': float(score),
        })
    return out

if not OPENAI_API_KEY:
    print('API_KEY_OPENAI não encontrado. Pulando etapa LLM.')
else:
    pending_idx = final_df.index[final_df['match_method'] == 'pending_llm'].tolist()
    print('Pending rows for LLM:', len(pending_idx))
    llm_calls = 0
    llm_matched = 0
    llm_no_match = 0
    started = time.time()
    for idx in pending_idx:
        if llm_calls >= MAX_LLM_CALLS:
            print('Atingiu MAX_LLM_CALLS, parando.')
            break
        row = final_df.loc[idx]
        candidates = top_candidates_for_row(row, limit=10)
        if not candidates:
            final_df.loc[idx, 'match_method'] = 'unmatched_no_candidates'
            llm_no_match += 1
            continue
        try:
            decision = llm_pick_candidate(row, candidates)
        except Exception:
            final_df.loc[idx, 'match_method'] = 'llm_error'
            final_df.loc[idx, 'match_score'] = None
            llm_no_match += 1
            continue
        llm_calls += 1
        match_id = decision.get('match_id_igcg', None)
        conf = decision.get('confidence', 0)
        try:
            conf = float(conf)
        except Exception:
            conf = 0.0
        valid_ids = {str(c['id_igcg']) for c in candidates if c.get('id_igcg') is not None}
        if match_id is None:
            final_df.loc[idx, 'match_method'] = 'llm_no_match'
            final_df.loc[idx, 'match_score'] = conf * 100
            llm_no_match += 1
            continue
        match_id_str = str(match_id).strip()
        if match_id_str.lower() in {'null', 'none', ''}:
            final_df.loc[idx, 'match_method'] = 'llm_no_match'
            final_df.loc[idx, 'match_score'] = conf * 100
            llm_no_match += 1
            continue
        if match_id_str not in valid_ids or conf < LLM_MIN_CONFIDENCE:
            final_df.loc[idx, 'match_method'] = 'llm_low_confidence'
            final_df.loc[idx, 'match_score'] = conf * 100
            llm_no_match += 1
            continue
        chosen = next((c for c in candidates if str(c['id_igcg']) == match_id_str), None)
        if not chosen:
            final_df.loc[idx, 'match_method'] = 'llm_invalid'
            final_df.loc[idx, 'match_score'] = conf * 100
            llm_no_match += 1
            continue
        final_df.loc[idx, 'id_igcg'] = match_id_str
        final_df.loc[idx, 'title_igcg'] = chosen.get('title_igcg')
        final_df.loc[idx, 'album_igcg'] = chosen.get('album_igcg')
        final_df.loc[idx, 'hymn_name'] = chosen.get('hymn_name')
        final_df.loc[idx, 'members'] = chosen.get('members')
        final_df.loc[idx, 'description'] = chosen.get('description')
        final_df.loc[idx, 'thumbnail'] = chosen.get('thumbnail')
        final_df.loc[idx, 'file_url'] = chosen.get('file_url')
        final_df.loc[idx, 'file_type'] = chosen.get('file_type')
        final_df.loc[idx, 'file_duration'] = chosen.get('file_duration')
        final_df.loc[idx, 'match_method'] = 'llm'
        final_df.loc[idx, 'match_score'] = conf * 100
        llm_matched += 1
        time.sleep(0.15)
    elapsed = time.time() - started
    print(f'LLM calls: {llm_calls}, matched: {llm_matched}, no-match: {llm_no_match}, elapsed: {elapsed:.1f}s')
    print('Total matched now:', final_df['id_igcg'].notna().sum(), 'of', len(final_df))

Pending rows for LLM: 966
Atingiu MAX_LLM_CALLS, parando.
LLM calls: 300, matched: 5, no-match: 295, elapsed: 371.7s
Total matched now: 346 of 1307


In [37]:
# 5) Relatórios para revisão
report_cols = [
    'id','slug','link','title','album_name','date','modified',
    'id_igcg','title_igcg','album_igcg','hymn_name','members','description','thumbnail','file_url','file_type','file_duration',
    'match_method','match_score',
]
existing = [c for c in report_cols if c in final_df.columns]
report = final_df[existing].copy()

print('Match method counts:')
print(report['match_method'].value_counts(dropna=False).head(30))

# amostras de casos sem match
unmatched = report[report['id_igcg'].isna()].copy()
unmatched[['title','album_name','match_method','match_score']].head(30)

Match method counts:
match_method
pending_llm     666
exact_title     310
llm_no_match    295
exact_combo      28
llm               5
fuzzy             3
Name: count, dtype: int64


,title,album_name,match_method,match_score
4,Deus é fiel,Nos Braços do Pai,llm_no_match,10.0
5,Jeremias 29-13,FVK,llm_no_match,10.0
7,Nossa Missão,NaN,llm_no_match,10.0
8,Encontro de Glorias,NaN,llm_no_match,10.0
13,A Fé vem pelo ouvir,Vida em Crescimento,llm_no_match,10.0
15,Ordem na Cidade – Hino novo,"Conf, Inter Fevereiro de 2025",llm_no_match,10.0
16,Time Vencedor,hino novo,llm_no_match,10.0
18,Vencendo vem Jesus,Cantor Cristão,llm_no_match,10.0
19,Carteiro,Hino novo,llm_no_match,10.0
20,De Deus não se zomba,Bom Depósito Vol.1,llm_no_match,10.0


In [ ]:
# 6) Exporta arquivos
OUT_ENRICHED = ROOT / 'songs_enriched_from_df_final.csv'
OUT_REPORT = ROOT / 'title_mapping_report.csv'

report.to_csv(OUT_REPORT, index=False, encoding='utf-8')

# dataset final para alimentar a tabela songs (mantém todas colunas originais do df_final e adiciona as enriquecidas)
enriched_cols = ['id_igcg','title_igcg','members','description','thumbnail','file_url','file_type','file_duration']
for c in enriched_cols:
    if c not in final_df.columns:
        final_df[c] = None

final_df.to_csv(OUT_ENRICHED, index=False, encoding='utf-8')

print('Wrote:', OUT_REPORT)
print('Wrote:', OUT_ENRICHED)

In [18]:
report

,id,slug,link,title,album_name,date,modified,id_igcg,title_igcg,hymn_name,members,description,thumbnail,file_url,file_type,file_duration,match_method,match_score
0,14166,10-tu-eres-mas-288,https://timcifras.com.br/musicas/10-tu-eres-ma...,Tú Eres Más,288,2023-04-11 20:12:41,2023-05-01 18:22:47,288-tu-eres-mas,CD 288 | Tú Eres Más,Tú Eres Más,288 Worship,EM BREVE,https://igcgcloud.netlify.app/images/cd288cons...,https://igcgcloud.netlify.app/CD-288/tueresmas...,audio/mp3,216.0,exact,100.000000
1,14429,whenever-im-low,https://timcifras.com.br/musicas/whenever-im-low/,Whenever I’m low,288,2023-04-11 20:14:07,2023-04-30 20:26:48,288-whenever-im-low,CD 288 | Whenever I'm Low,Whenever I'm Low,288 Worship,EM BREVE,https://igcgcloud.netlify.app/images/cd288cons...,https://igcgcloud.netlify.app/CD-288/wheneveri...,audio/mp3,122.0,exact,100.000000
2,17173,nao-ha-ninguem-igual-pra-mim-cd-oh-deus-abenco...,https://timcifras.com.br/musicas/nao-ha-ningue...,Não há ninguém igual pra mim,Oh Deus abençoe a Àfrica,2025-09-23 20:57:54,2025-09-23 20:57:54,daaa-nao-ha-ninguem-igual-pra-mim,CD Deus Abençoe a África | Não Há Nínguem Igua...,Não Há Nínguem Igual Pra Mim,EAV,EM BREVE,https://igcgcloud.netlify.app/images/capacddeu...,https://igcgcloud.netlify.app/cd-deusabencoeaa...,audio/mp3,230.0,exact,100.000000
3,17159,antes-de-haver-o-tempo,https://timcifras.com.br/musicas/antes-de-have...,Antes de Haver o Tempo,NaN,2025-09-08 20:59:25,2025-09-09 11:19:44,288-antes-de-haver-o-tempo,CD 288 | Antes de Haver o Tempo,Antes de Haver o Tempo,288 Worship,,https://igcgcloud.netlify.app/images/capaantes...,https://igcgcloud.netlify.app/CD-288/antesdeha...,audio/mp3,288.0,exact,100.000000
4,17151,deus-e-fiel-cd-nos-bracos-do-pai-sou-um-vaso-p...,https://timcifras.com.br/musicas/deus-e-fiel-c...,Deus é fiel,Nos Braços do Pai,2025-08-27 18:50:30,2025-08-27 19:12:36,None,None,None,None,None,None,None,None,NaN,unmatched_low_score,85.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,14051,ele-me-mostrou-cd-no-santo-dos-santos,https://timcifras.com.br/musicas/ele-me-mostro...,Ele Me Mostrou,No Santo dos Santos,2023-04-11 20:12:14,2023-10-09 10:11:28,vev-ele-me-mostrou,CD Voz e Violão | Ele me Mostrou,Ele me Mostrou,--,EM BREVE,https://igcgcloud.netlify.app/images/default.png,https://igcgcloud.netlify.app/cd-vozeviolao/el...,audio/mp3,231.0,exact,100.000000
1346,14050,02-vestigios-desse-amor-288,https://timcifras.com.br/musicas/02-vestigios-...,Vestígios Desse Amor,02,2023-04-11 20:12:14,2023-04-30 20:17:22,288-vestigios-desse-amor,CD 288 | Vestígios Desse Amor,Vestígios Desse Amor,288 Worship,EM BREVE,https://igcgcloud.netlify.app/images/cd288cons...,https://igcgcloud.netlify.app/CD-288/vestigios...,audio/mp3,300.0,exact,100.000000
1347,14049,hino-02,https://timcifras.com.br/musicas/hino-02/,Das bênçãos fontes é Deus Pai,Hino 02,2023-04-11 20:12:14,2023-05-01 21:47:49,None,None,None,None,None,None,None,None,NaN,unmatched_low_score,85.500000
1348,14048,hino-03,https://timcifras.com.br/musicas/hino-03/,Hino 03,NaN,2023-04-11 20:12:14,2023-04-11 20:12:14,None,None,None,None,None,None,None,None,NaN,unmatched_low_score,69.090909
